# CEG-WM Stage A2 HF one-click Colab execution

Run all cells after setting Colab Secrets `CEG_WM_ROOT_KEY` and `HF_TOKEN`. The notebook resolves the current Stage-A branch head once, checks it out detached, and automatically reuses a verified final package or the highest compatible checkpoint. Evidence remains incomplete/not evaluated: the three preregistered attacks and LPIPS are absent, and model revision/weight digests are not recorded.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
import os
def _required_secret(name):
    value = os.environ.pop(name, None)
    if value is None:
        value = userdata.get(name)
    if not isinstance(value, str) or not value:
        raise RuntimeError(f'missing required Colab Secret: {name}')
    return value
root_key = _required_secret('CEG_WM_ROOT_KEY')
hf_token = _required_secret('HF_TOKEN')
run_store_root = Path('/content/drive/MyDrive/CEG-WM/stage_a2_hf')
run_store_root.mkdir(parents=True, exist_ok=True)


In [ ]:
import json, re, subprocess, sys
repo = Path('/content/CEG-WM-stage-a-exact')
if repo.exists():
    raise RuntimeError('detached checkout path already exists')
subprocess.run(['git', 'init', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'remote', 'add', 'origin', 'https://github.com/RICHAAARC/CEG-WM.git'], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', 'refs/heads/stage-a-method-feasibility'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
resolved_exact = subprocess.run(['git', '-C', str(repo), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if re.fullmatch(r'[0-9a-f]{40}', resolved_exact) is None:
    raise RuntimeError('resolved Stage-A branch head is not an exact revision')
if subprocess.run(['git', '-C', str(repo), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout:
    raise RuntimeError('execution checkout is not clean')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo)], check=True)


In [ ]:
local_output_root = Path('/content/cegwm-stage-a2-local')
runner_env = dict(os.environ)
runner_env['CEG_WM_ROOT_KEY'] = root_key
runner_env['HF_TOKEN'] = hf_token
command = [sys.executable, '-m', 'experiments.stage_a.run_hf_a2_colab', '--repo-root', str(repo), '--output-root', str(local_output_root), '--expected-exact', resolved_exact, '--run-store-root', str(run_store_root)]
run_id = None
try:
    process = subprocess.Popen(command, env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    for line in process.stdout:
        if line.startswith('CEGWM_PROGRESS '):
            progress = json.loads(line.removeprefix('CEGWM_PROGRESS '))
            candidate_run_id = progress.get('run_id')
            if re.fullmatch(r'a2hf-[0-9a-f]{24}', candidate_run_id or '') is None:
                raise RuntimeError('runner progress has invalid deterministic run identity')
            if run_id is not None and run_id != candidate_run_id:
                raise RuntimeError('runner changed deterministic run identity')
            run_id = candidate_run_id
            print({'run_id': run_id, 'committed': progress['committed'], 'fixed_total': progress['fixed_total']})
    runner_rc = process.wait()
finally:
    runner_env.pop('CEG_WM_ROOT_KEY', None)
    runner_env.pop('HF_TOKEN', None)
    root_key = hf_token = ''
    del root_key, hf_token, runner_env
if run_id is None:
    raise RuntimeError('runner produced no deterministic run identity')


In [ ]:
import hashlib, zipfile
drive_run_dir = run_store_root / run_id
zip_path = drive_run_dir / f'{run_id}.zip'
checksum_path = drive_run_dir / f'{run_id}.zip.sha256'
if not (zip_path.is_file() and checksum_path.is_file()):
    raise RuntimeError('runner did not publish a complete final package pair')
checksum_parts = checksum_path.read_text().strip().split()
if len(checksum_parts) != 2 or checksum_parts[1] != zip_path.name:
    raise RuntimeError('final checksum file is malformed')
zip_sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
if checksum_parts[0] != zip_sha256:
    raise RuntimeError('final ZIP checksum mismatch')
with zipfile.ZipFile(zip_path) as archive:
    if set(archive.namelist()) != {'receipt.json', 'result.json'}:
        raise RuntimeError('final ZIP members mismatch')
    receipt = json.loads(archive.read('receipt.json'))
    result = json.loads(archive.read('result.json'))
if receipt['run_id'] != run_id or result['run_id'] != run_id:
    raise RuntimeError('final run identity mismatch')
if receipt['resolved_exact'] != resolved_exact or result['resolved_exact'] != resolved_exact:
    raise RuntimeError('final exact mismatch')
if receipt['rc'] != runner_rc or result['rc'] != runner_rc:
    raise RuntimeError('runner/final RC mismatch')
if result['fixed_unit_count'] != 8 or result['fixed_record_count'] != 16 or len(result['records']) != 16:
    raise RuntimeError('final fixed denominator mismatch')
summary = {'run_id': run_id, 'resolved_exact': resolved_exact, 'status': receipt['status'], 'rc': runner_rc, 'drive_relative_dir': f'CEG-WM/stage_a2_hf/{run_id}', 'zip_sha256': zip_sha256}
print(summary)
if runner_rc != 0:
    raise RuntimeError('runner completed with retained operational failures')
